# PRCP-1017-AutoPricePred — Clean Complete Project

This single notebook covers EDA, cleaning, multiple regression models, model comparison, cross-validation, tuning, evaluation, feature importance, business analysis, challenges, prediction, and model export.

In [ ]:

import sys, zipfile, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import joblib

from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error, make_scorer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
print("Python:", sys.version.split()[0])
import sklearn
print("scikit-learn:", sklearn.__version__)

In [ ]:
# Compatibility check
# This project supports both older and newer scikit-learn versions.
import sklearn
print("scikit-learn version:", sklearn.__version__)

try:
    test_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    print("OneHotEncoder API: sparse_output supported")
except TypeError:
    test_encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)
    print("OneHotEncoder API: legacy sparse supported")


In [ ]:
DATASET_URL = "https://d3ilbtxij3aepc.cloudfront.net/projects/CDS-Capstone-Projects/PRCP-1017-AutoPricePred.zip"
DATA_DIR = Path("PRCP-1017-AutoPricePred_data")
DATA_DIR.mkdir(exist_ok=True)
ZIP_PATH = DATA_DIR / "PRCP-1017-AutoPricePred.zip"

if not list(DATA_DIR.rglob("*.csv")):
    print("Downloading dataset...")
    r = requests.get(DATASET_URL, timeout=120)
    r.raise_for_status()
    ZIP_PATH.write_bytes(r.content)
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(DATA_DIR)

csvs = list(DATA_DIR.rglob("*.csv"))
if not csvs:
    raise FileNotFoundError("No CSV file found in the extracted project.")
train_csvs = [p for p in csvs if "train" in p.name.lower()]
csv_path = train_csvs[0] if train_csvs else csvs[0]
print("Using:", csv_path)


In [ ]:
columns = [
"symboling","normalized-losses","make","fuel-type","aspiration","num-of-doors",
"body-style","drive-wheels","engine-location","wheel-base","length","width",
"height","curb-weight","engine-type","num-of-cylinders","engine-size",
"fuel-system","bore","stroke","compression-ratio","horsepower","peak-rpm",
"city-mpg","highway-mpg","price"
]

raw = pd.read_csv(csv_path, header=None, na_values=["?","NA","N/A",""])
first = raw.iloc[0].astype(str).str.strip().str.lower().tolist()

if "price" in first and "make" in first:
    df = pd.read_csv(csv_path, na_values=["?","NA","N/A",""])
else:
    if raw.shape[1] != len(columns):
        raise ValueError(f"Expected {len(columns)} columns but found {raw.shape[1]}.")
    df = raw.copy()
    df.columns = columns

df.columns = (df.columns.astype(str).str.strip().str.lower()
              .str.replace("_","-",regex=False).str.replace(" ","-",regex=False))

numeric_columns = [
"symboling","normalized-losses","wheel-base","length","width","height",
"curb-weight","engine-size","bore","stroke","compression-ratio",
"horsepower","peak-rpm","city-mpg","highway-mpg","price"
]

for c in numeric_columns:
    df[c] = pd.to_numeric(df[c], errors="coerce")

for c in df.select_dtypes(include=["object","string"]).columns:
    df[c] = df[c].astype("string").str.strip()

df = df.dropna(subset=["price"]).drop_duplicates().reset_index(drop=True)

print("Shape:", df.shape)
display(df.head())


In [ ]:
# Complete data-quality report
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_%": (df.isna().mean()*100).round(2),
    "unique": df.nunique()
}).sort_values("missing", ascending=False))

print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)


In [ ]:
# Outlier report
rows = []
for c in df.select_dtypes(include=np.number).columns:
    q1, q3 = df[c].quantile([.25,.75])
    iqr = q3-q1
    low, high = q1-1.5*iqr, q3+1.5*iqr
    count = ((df[c]<low)|(df[c]>high)).sum()
    rows.append([c,q1,q3,iqr,low,high,count])
outlier_report = pd.DataFrame(rows, columns=["Feature","Q1","Q3","IQR","Lower","Upper","Outlier_Count"])
display(outlier_report.sort_values("Outlier_Count", ascending=False))


In [ ]:
# Target analysis
print(df["price"].describe())
plt.figure(figsize=(10,5))
sns.histplot(df["price"], kde=True)
plt.title("Price Distribution")
plt.show()

plt.figure(figsize=(10,4))
sns.boxplot(x=df["price"])
plt.title("Price Boxplot")
plt.show()


In [ ]:
# Categorical EDA
cat_eda = ["make","fuel-type","aspiration","num-of-doors","body-style",
           "drive-wheels","engine-location","engine-type","num-of-cylinders","fuel-system"]
for c in cat_eda:
    plt.figure(figsize=(10,5))
    order = df.groupby(c)["price"].mean().sort_values(ascending=False).index
    sns.boxplot(data=df, x=c, y="price", order=order)
    plt.title(f"Price by {c}")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# Numerical EDA and correlation
num_eda = [c for c in numeric_columns if c != "price"]
for c in num_eda:
    plt.figure(figsize=(7,4))
    sns.scatterplot(data=df, x=c, y="price")
    plt.title(f"{c} vs Price")
    plt.tight_layout()
    plt.show()

corr = df.select_dtypes(include=np.number).corr()
plt.figure(figsize=(15,11))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()
display(corr["price"].sort_values(ascending=False).to_frame("Correlation_with_Price"))


In [ ]:
# Business EDA
print("Average price by manufacturer")
display(df.groupby("make")["price"].agg(["count","mean","median"]).sort_values("mean",ascending=False))
print("Average price by body style")
display(df.groupby("body-style")["price"].agg(["count","mean","median"]).sort_values("mean",ascending=False))
print("Average price by drive wheels")
display(df.groupby("drive-wheels")["price"].agg(["count","mean","median"]).sort_values("mean",ascending=False))


In [ ]:
# Train/test split
X = df.drop(columns=["price"])
y = df["price"].astype(float)
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.20,random_state=RANDOM_STATE)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print(X_train.shape, X_test.shape)


In [ ]:
# Version-safe OneHotEncoder and preprocessing
def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def linear_preprocessor():
    return ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_encoder())
        ]), categorical_features)
    ])

def tree_preprocessor():
    return ColumnTransformer([
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", make_encoder())
        ]), categorical_features)
    ])

print("Preprocessing ready.")


In [ ]:
# Models
models = {
"Linear Regression": Pipeline([("preprocessor",linear_preprocessor()),("model",LinearRegression())]),
"Ridge Regression": Pipeline([("preprocessor",linear_preprocessor()),("model",Ridge(alpha=10))]),
"Lasso Regression": Pipeline([("preprocessor",linear_preprocessor()),("model",Lasso(alpha=.01,max_iter=50000))]),
"Elastic Net": Pipeline([("preprocessor",linear_preprocessor()),("model",ElasticNet(alpha=.01,l1_ratio=.5,max_iter=50000))]),
"Random Forest": Pipeline([("preprocessor",tree_preprocessor()),("model",RandomForestRegressor(n_estimators=500,random_state=RANDOM_STATE,n_jobs=-1))]),
"Extra Trees": Pipeline([("preprocessor",tree_preprocessor()),("model",ExtraTreesRegressor(n_estimators=500,random_state=RANDOM_STATE,n_jobs=-1))]),
"Gradient Boosting": Pipeline([("preprocessor",tree_preprocessor()),("model",GradientBoostingRegressor(n_estimators=300,learning_rate=.03,max_depth=3,random_state=RANDOM_STATE))]),
"Hist Gradient Boosting": Pipeline([("preprocessor",tree_preprocessor()),("model",HistGradientBoostingRegressor(max_iter=300,learning_rate=.05,max_leaf_nodes=15,random_state=RANDOM_STATE))]),
"SVR": Pipeline([("preprocessor",linear_preprocessor()),("model",SVR(kernel="rbf",C=100,epsilon=.05))])
}
print("Models:", list(models))


In [ ]:
# Baseline
baseline_pred = np.full(len(y_test), y_train.mean())
print("Baseline MAE:", mean_absolute_error(y_test,baseline_pred))
print("Baseline RMSE:", np.sqrt(mean_squared_error(y_test,baseline_pred)))
print("Baseline R2:", r2_score(y_test,baseline_pred))


In [ ]:
# 15. Holdout Model Comparison — FIXED
# The original error occurs at model.fit(X_train, y_train). The exact failing
# estimator is hidden by the loop. This version tests every model separately
# and records the full error instead of stopping the notebook.
#
# The most common cause in this project is a scikit-learn compatibility issue
# around OneHotEncoder(sparse_output=...) versus sparse=..., which is already
# handled by make_encoder() above.

holdout_rows = []
model_errors = []
trained_models = {}

for name, model in models.items():
    print(f"\nTraining: {name}")
    try:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)

        holdout_rows.append([
            name,
            mean_absolute_error(y_test, pred),
            np.sqrt(mean_squared_error(y_test, pred)),
            r2_score(y_test, pred),
            mean_absolute_percentage_error(y_test, pred)
        ])

        trained_models[name] = model
        print("  SUCCESS")

    except Exception as e:
        model_errors.append([
            name,
            type(e).__name__,
            str(e)
        ])
        print("  FAILED")
        print("  Error type:", type(e).__name__)
        print("  Error:", e)

holdout_df = pd.DataFrame(
    holdout_rows,
    columns=["Model", "MAE", "RMSE", "R2", "MAPE"]
).sort_values("RMSE").reset_index(drop=True)

print("\nMODEL COMPARISON")
display(holdout_df)

if model_errors:
    print("\nMODELS WITH ERRORS")
    display(pd.DataFrame(
        model_errors,
        columns=["Model", "Error_Type", "Error_Message"]
    ))

if holdout_df.empty:
    raise RuntimeError(
        "All models failed. Read the MODEL WITH ERRORS table above for "
        "the exact exception."
    )


In [ ]:
# Model comparison charts
if len(holdout_df):
    plt.figure(figsize=(10,6))
    sns.barplot(data=holdout_df,x="RMSE",y="Model")
    plt.title("RMSE Comparison — Lower is Better")
    plt.show()

    plt.figure(figsize=(10,6))
    sns.barplot(data=holdout_df,x="R2",y="Model")
    plt.title("R2 Comparison — Higher is Better")
    plt.show()


In [ ]:
# Version-independent 5-fold CV scorers
def rmse(y_true,y_pred):
    return np.sqrt(mean_squared_error(y_true,y_pred))

rmse_scorer = make_scorer(rmse, greater_is_better=False)
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)
r2_scorer = make_scorer(r2_score)

kf = KFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)

cv_rows, cv_errors = [], []
for name, model in models.items():
    print("CV:", name)
    try:
        s = cross_validate(model,X_train,y_train,cv=kf,
            scoring={"MAE":mae_scorer,"RMSE":rmse_scorer,"R2":r2_scorer},
            n_jobs=-1,error_score="raise")
        cv_rows.append([name,-s["test_MAE"].mean(),-s["test_RMSE"].mean(),
                        s["test_R2"].mean(),s["test_RMSE"].std()])
    except Exception as e:
        cv_errors.append([name,type(e).__name__,str(e)])

cv_df = pd.DataFrame(cv_rows,columns=["Model","CV_MAE","CV_RMSE","CV_R2","CV_RMSE_STD"]).sort_values("CV_RMSE").reset_index(drop=True)
display(cv_df)
if cv_errors:
    display(pd.DataFrame(cv_errors,columns=["Model","Error_Type","Message"]))


In [ ]:
# Best model
if cv_df.empty:
    raise RuntimeError("No model completed CV. See the CV error table above.")
best_model_name = cv_df.iloc[0]["Model"]
print("Best model:", best_model_name)


In [ ]:
# Hyperparameter tuning
grids = {
"Ridge Regression":{"model__alpha":[.1,1,10,50,100]},
"Lasso Regression":{"model__alpha":[.001,.005,.01,.05,.1]},
"Elastic Net":{"model__alpha":[.001,.005,.01,.05],"model__l1_ratio":[.2,.5,.8]},
"Random Forest":{"model__n_estimators":[300,500],"model__max_depth":[None,10,20],"model__min_samples_leaf":[1,2]},
"Extra Trees":{"model__n_estimators":[300,500],"model__max_depth":[None,10,20],"model__min_samples_leaf":[1,2]},
"Gradient Boosting":{"model__n_estimators":[200,300],"model__learning_rate":[.03,.05],"model__max_depth":[2,3]},
"Hist Gradient Boosting":{"model__max_iter":[200,300],"model__learning_rate":[.03,.05],"model__max_leaf_nodes":[10,15,20]},
"SVR":{"model__C":[10,50,100],"model__epsilon":[.03,.05,.1],"model__gamma":["scale","auto"]}
}

if best_model_name in grids:
    search = GridSearchCV(models[best_model_name],grids[best_model_name],
                          scoring=rmse_scorer,cv=kf,n_jobs=-1,refit=True,error_score="raise")
    search.fit(X_train,y_train)
    production_model = search.best_estimator_
    print("Best parameters:",search.best_params_)
    print("Best CV RMSE:",-search.best_score_)
else:
    production_model = models[best_model_name]
    production_model.fit(X_train,y_train)


In [ ]:
# Final evaluation
final_pred = production_model.predict(X_test)
final_metrics = pd.DataFrame({
"Metric":["MAE","RMSE","R2","MAPE"],
"Value":[mean_absolute_error(y_test,final_pred),
         np.sqrt(mean_squared_error(y_test,final_pred)),
         r2_score(y_test,final_pred),
         mean_absolute_percentage_error(y_test,final_pred)]
})
display(final_metrics)


In [ ]:
# Actual vs predicted and residuals
prediction_results = pd.DataFrame({"Actual_Price":y_test.values,"Predicted_Price":final_pred})
prediction_results["Absolute_Error"]=(prediction_results["Actual_Price"]-prediction_results["Predicted_Price"]).abs()
prediction_results["Residual"]=prediction_results["Actual_Price"]-prediction_results["Predicted_Price"]
display(prediction_results.head(20))

plt.figure(figsize=(8,6))
sns.scatterplot(data=prediction_results,x="Actual_Price",y="Predicted_Price")
lo=min(prediction_results.min()); hi=max(prediction_results.max())
plt.plot([lo,hi],[lo,hi],"--")
plt.title("Actual vs Predicted Price")
plt.show()

plt.figure(figsize=(8,5))
sns.scatterplot(data=prediction_results,x="Predicted_Price",y="Residual")
plt.axhline(0,ls="--")
plt.title("Residuals vs Predicted Price")
plt.show()


In [ ]:
# Feature importance
perm = permutation_importance(production_model,X_test,y_test,n_repeats=10,
                              random_state=RANDOM_STATE,scoring=rmse_scorer,n_jobs=-1)
feature_importance = pd.DataFrame({
"Feature":X_test.columns,
"Importance":perm.importances_mean,
"Importance_STD":perm.importances_std
}).sort_values("Importance",ascending=False)
display(feature_importance)

plt.figure(figsize=(9,8))
sns.barplot(data=feature_importance.head(20),x="Importance",y="Feature")
plt.title("Top 20 Features — Permutation Importance")
plt.tight_layout()
plt.show()


In [ ]:
# Example prediction
example_car = pd.DataFrame([{
"symboling":1,"normalized-losses":np.nan,"make":"toyota","fuel-type":"gas",
"aspiration":"std","num-of-doors":"four","body-style":"sedan","drive-wheels":"fwd",
"engine-location":"front","wheel-base":100.0,"length":180.0,"width":68.0,
"height":54.0,"curb-weight":2600,"engine-type":"ohc","num-of-cylinders":"four",
"engine-size":120,"fuel-system":"mpfi","bore":3.2,"stroke":3.2,
"compression-ratio":9.0,"horsepower":90,"peak-rpm":5500,"city-mpg":30,
"highway-mpg":35
}])[X.columns]

print(f"Example predicted price: ${production_model.predict(example_car)[0]:,.2f}")

def predict_car_price(car_data):
    missing = sorted(set(X.columns)-set(car_data.columns))
    if missing:
        raise ValueError(f"Missing features: {missing}")
    return production_model.predict(car_data[X.columns])



## Business Analysis / Management Recommendations

- Use the model as a data-driven price-estimation tool for new vehicle concepts.
- Study high-importance specifications when designing economy, mid-range, and premium variants.
- Compare predicted price with manufacturing cost and target margin before final pricing.
- Use manufacturer, body style, drivetrain, engine specifications, dimensions, and fuel economy as inputs to market-positioning decisions.
- Review low-importance expensive features for possible cost optimization.
- Retrain the model periodically with current market data.

**Important:** feature importance indicates predictive association, not causation.


## Challenges Faced and Techniques Used

1. **Missing values:** median imputation for numerical variables and most-frequent imputation for categorical variables.
2. **`?` values:** converted to `NaN` so they can be handled correctly.
3. **Categorical variables:** One-Hot Encoding with `handle_unknown='ignore'`.
4. **Different feature scales:** StandardScaler for scale-sensitive models.
5. **Outliers:** IQR analysis; genuine expensive cars were not automatically deleted.
6. **Overfitting:** train/test split, 5-fold cross-validation, and hyperparameter tuning.
7. **Data leakage:** all preprocessing is inside `Pipeline`/`ColumnTransformer`.
8. **Model selection:** several regression algorithms were compared.
9. **Interpretability:** permutation feature importance.
10. **Version compatibility:** OneHotEncoder is created with a try/except for `sparse_output` versus `sparse`, preventing the common TypeError.


In [ ]:
# Save final artifacts
joblib.dump(production_model,"PRCP-1017-AutoPricePred_production_model.joblib")
holdout_df.to_csv("PRCP-1017-AutoPricePred_holdout_model_comparison.csv",index=False)
cv_df.to_csv("PRCP-1017-AutoPricePred_cross_validation_model_comparison.csv",index=False)
feature_importance.to_csv("PRCP-1017-AutoPricePred_feature_importance.csv",index=False)
prediction_results.to_csv("PRCP-1017-AutoPricePred_test_predictions.csv",index=False)

print("Project completed successfully.")
print("Production model:",best_model_name)
